To plot the results we need the real estimates.

Run the cells in this Notebook to generate the file containing the correct results of each experiment.

In [ ]:
from datetime import datetime
import duckdb
import pandas as pd
from pathlib import Path

In [ ]:
# The path to the folder containing all the datasets used in the experiment.
# I'm optionally using converted parquet files due to overall increase in performance. Change this as needed.
dataset_folder = "../data/parquet"

# The path to the folder containing all the input files for the experiments.
input_folder = Path("../in")

In [ ]:
def read_experiment_config(file_path):
    # First line is the dataset filename (in CSV).
    with open(file_path, 'r') as f:
        lines = f.readlines()
    dataset_file = lines[0].strip()
    
    # Subsequent lines are the predicates (experiments; one per line).
    predicates = [line.strip() for line in lines[1:]]
    return dataset_file, predicates

In [ ]:
def parse_predicate(predicate_str):
  conditions = predicate_str.split(' AND ')
  parsed_conditions = []
  
  for condition in conditions:
    condition = condition.strip()
    
    if ' = ' in condition:
      left, right = condition.split(' = ')
      left = left.strip()
      right = right.strip()
      parsed_conditions.append(f't1.{left} = t2.{right}')
    elif ' != ' in condition or ' <> ' in condition:
      if ' != ' in condition:
        left, right = condition.split(' != ')
      else:
        left, right = condition.split(' <> ')
      left = left.strip()
      right = right.strip()
      parsed_conditions.append(f't1.{left} != t2.{right}')
  
  return ' AND '.join(parsed_conditions)

def compute_real_estimate(conn, table_name, predicate):
  query = f"""
  SELECT COUNT(*) as result_size
  FROM {table_name} t1, {table_name} t2
  WHERE {parse_predicate(predicate)}
  """
  result = conn.execute(query).fetchone()
  return result[0]


In [ ]:
# PS: This cell may take a long time to execute.

datasets = list(input_folder.glob("*"))

results = []

for experiment_path in datasets:
  dataset_fname, predicates = read_experiment_config(experiment_path)
  parquet_path = Path(dataset_folder) / dataset_fname.replace(".csv", ".parquet")
  
  # DEBUG
  # Skipping some large datasets for now
  if "4000000" in parquet_path.name:
    continue

  print(f"{datetime.now().time()} -> Processing {parquet_path.name}")
  
  conn = duckdb.connect(database=":memory:")
  conn.execute(f"""
    CREATE TABLE tmp_data AS SELECT * FROM read_parquet('{parquet_path}')
  """)

  for predicate in predicates:
    real_estimate = compute_real_estimate(conn, "tmp_data", predicate)

    results.append({
      "experiment_fname": experiment_path.name,
      "dataset_fname": dataset_fname,
      "predicate": predicate,
      "real_estimate": real_estimate
    })

  conn.close()

real_estimates = pd.DataFrame(results)

18:50:10.804699 -> Processing flights_1000000.parquet
18:50:34.338015 -> Processing flights_100000.parquet
18:50:35.646285 -> Processing flights_10000.parquet
18:50:35.688284 -> Processing flights_500000.parquet
18:50:49.463504 -> Processing gen0_100000.parquet
18:51:12.485773 -> Processing gen0_10000.parquet
18:51:12.738964 -> Processing gen0_1000000.parquet
18:57:57.409347 -> Processing gen0_2000000.parquet
19:25:52.613377 -> Processing gen0_500000.parquet
19:28:40.216921 -> Processing gen1_100000.parquet
19:29:02.868598 -> Processing gen1_10000.parquet
19:29:03.120078 -> Processing gen1_1000000.parquet
19:35:51.159032 -> Processing gen1_2000000.parquet
20:03:05.827741 -> Processing gen1_500000.parquet
20:05:31.264091 -> Processing titlekind_1000000.parquet
20:06:15.052439 -> Processing titlekind_100000.parquet
20:06:17.616075 -> Processing titlekind_10000.parquet
20:06:17.669046 -> Processing titlekind_500000.parquet
20:06:33.993058 -> Processing tax_1000000.parquet
20:06:50.981471 

In [ ]:
real_estimates.to_csv("../results/real_estimates.csv", index=False)